# The AI Trust Gap — FIXED 2025 Pipeline

This version avoids the broken ZIP/raw-URL download path.

It retrieves the official **Stack Overflow Developer Survey 2025** files from the
`StackExchange/Survey` GitHub repository using Git + Git LFS, then performs the
respondent-level analysis.


In [ ]:
# 1. Setup
from pathlib import Path
import subprocess, sys, os, warnings
warnings.filterwarnings("ignore")

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
DATA = ROOT / "ai_trust_gap_data"
OUTPUT = ROOT / "outputs"
FIGURES = OUTPUT / "figures"

DATA.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("OUTPUT:", OUTPUT)


In [ ]:
# 2. Install Git LFS and Python dependencies
subprocess.run(["apt-get","update","-qq"], check=False)
subprocess.run(["apt-get","install","-y","-qq","git-lfs"], check=True)
subprocess.run(["git","lfs","install"], check=True)

packages = ["pandas","numpy","matplotlib","scipy","statsmodels"]
subprocess.run([sys.executable,"-m","pip","install","-q",*packages], check=True)

print("Dependencies ready.")


In [ ]:
# 3. Download ONLY the official 2025 survey directory using sparse Git checkout
REPO = ROOT / "StackExchange_Survey"

if not REPO.exists():
    subprocess.run([
        "git","clone",
        "--depth","1",
        "--filter=blob:none",
        "--sparse",
        "https://github.com/StackExchange/Survey.git",
        str(REPO)
    ], check=True)

subprocess.run(
    ["git","-C",str(REPO),"sparse-checkout","set","packages/archive/2025"],
    check=True
)

# Explicitly fetch the large LFS survey response file
subprocess.run(
    ["git","-C",str(REPO),"lfs","pull","--include=packages/archive/2025/results.csv"],
    check=True
)

RAW_PATH = REPO / "packages/archive/2025/results.csv"
SCHEMA_PATH = REPO / "packages/archive/2025/schema.csv"

print("results.csv exists:", RAW_PATH.exists())
print("schema.csv exists:", SCHEMA_PATH.exists())

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Could not find {RAW_PATH}")
if not SCHEMA_PATH.exists():
    raise FileNotFoundError(f"Could not find {SCHEMA_PATH}")

print("2025 data ready.")


In [ ]:
# 4. Verify that results.csv is the actual CSV, not an LFS pointer
print("results.csv size (MB):", round(RAW_PATH.stat().st_size / 1024 / 1024, 1))

with open(RAW_PATH, "rb") as f:
    first_bytes = f.read(100)

if b"git-lfs.github.com/spec" in first_bytes:
    raise RuntimeError("Git LFS pointer detected instead of real data file.")

print("File verification passed.")


In [ ]:
# 5. Load schema and inspect actual 2025 columns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

schema = pd.read_csv(SCHEMA_PATH)
raw_columns = pd.read_csv(RAW_PATH, nrows=0).columns.tolist()

print("Schema shape:", schema.shape)
print("Number of response columns:", len(raw_columns))
display(schema.head())
print(raw_columns)


## Variable resolver

The code below discovers which candidate AI and professional-context variables are actually present in the 2025 file.


In [ ]:
# 6. Resolve relevant fields dynamically
candidate_groups = {
    "id": ["ResponseId"],
    "ai_use": ["AISelect"],
    "ai_sentiment": ["AISent"],
    "ai_accuracy": ["AIAcc"],
    "ai_complex": ["AIComplex"],
    "ai_threat": ["AIThreat"],
    "ai_benefit": ["AIBen"],
    "ai_agents": ["AIAgents"],
    "ai_models": ["AIModelsChoice"],
    "work_exp": ["WorkExp"],
    "years_code": ["YearsCode"],
    "age": ["Age"],
    "country": ["Country"],
    "dev_type": ["DevType"],
    "industry": ["Industry"],
    "employment": ["Employment"],
    "remote_work": ["RemoteWork"],
    "org_size": ["OrgSize"],
    "ic_pm": ["ICorPM"],
    "education": ["EdLevel"],
    "job_sat": ["JobSat"],
}

resolved = {}
for concept, candidates in candidate_groups.items():
    resolved[concept] = next((c for c in candidates if c in raw_columns), None)

resolved_df = pd.DataFrame(
    [{"concept":k,"column":v,"available":v is not None} for k,v in resolved.items()]
)
display(resolved_df)

usecols = sorted({v for v in resolved.values() if v is not None})
print("Loading", len(usecols), "relevant columns.")


In [ ]:
# 7. Load respondent-level data
df = pd.read_csv(RAW_PATH, usecols=usecols, low_memory=False)
rename_map = {v:k for k,v in resolved.items() if v is not None}
d = df.rename(columns=rename_map).copy()

for c in d.select_dtypes(include="object").columns:
    d[c] = d[c].replace(r"^\s*$", np.nan, regex=True)

for c in ["work_exp","years_code","job_sat"]:
    if c in d:
        d[c] = pd.to_numeric(d[c], errors="coerce")

print("Loaded respondent-level frame:", d.shape)
display(d.head())


In [ ]:
# 8. Inspect AI field categories before recoding
for c in ["ai_use","ai_sentiment","ai_accuracy","ai_complex","ai_agents","ic_pm"]:
    if c in d:
        print("\n---", c, "---")
        display(d[c].value_counts(dropna=False).head(20))


In [ ]:
# 9. Engineer research variables
if "work_exp" in d:
    d["experience_band"] = pd.cut(
        d["work_exp"],
        bins=[-np.inf,5,10,np.inf],
        labels=["Early career (≤5)","Mid career (6–10)","Experienced (10+)"]
    )

if "ai_accuracy" in d:
    s = d["ai_accuracy"].astype("string").str.lower()
    d["trust_group"] = np.select(
        [
            s.str.contains("trust", na=False) & ~s.str.contains("distrust", na=False),
            s.str.contains("distrust", na=False),
        ],
        ["Trust","Distrust"],
        default="Neutral / unsure / missing"
    )

print("Engineered columns ready.")


In [ ]:
# 10. Data-quality report
quality = pd.DataFrame({
    "column": d.columns,
    "non_null": [d[c].notna().sum() for c in d.columns],
    "missing_pct": [round(d[c].isna().mean()*100,1) for c in d.columns],
    "unique": [d[c].nunique(dropna=True) for c in d.columns]
})
quality.to_csv(OUTPUT/"data_quality_report.csv", index=False)
display(quality)


In [ ]:
# 11. AI use vs trust
if {"ai_use","trust_group"}.issubset(d.columns):
    tab = pd.crosstab(d["ai_use"], d["trust_group"], normalize="index")*100
    tab.to_csv(OUTPUT/"ai_use_vs_trust_pct.csv")
    display(tab.round(1))

    ax = tab.plot(kind="bar", figsize=(11,6))
    ax.set_ylabel("Percent")
    ax.set_title("AI usage vs trust in AI accuracy")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURES/"01_ai_use_vs_trust.png", dpi=160)
    plt.show()


In [ ]:
# 12. Experience vs trust + chi-square
if {"experience_band","trust_group"}.issubset(d.columns):
    tab = pd.crosstab(d["experience_band"], d["trust_group"], normalize="index")*100
    tab.to_csv(OUTPUT/"experience_vs_trust_pct.csv")
    display(tab.round(1))

    contingency = pd.crosstab(d["experience_band"], d["trust_group"])
    chi2,p,dof,expected = stats.chi2_contingency(contingency)
    n = contingency.to_numpy().sum()
    r,k = contingency.shape
    v = np.sqrt((chi2/n)/max(1,min(k-1,r-1)))
    print(f"Chi-square={chi2:.2f}, p={p:.4g}, Cramer's V={v:.3f}")

    ax = tab.plot(kind="bar", figsize=(10,6))
    ax.set_ylabel("Percent")
    ax.set_title("Trust in AI accuracy by professional experience")
    plt.xticks(rotation=25,ha="right")
    plt.tight_layout()
    plt.savefig(FIGURES/"02_experience_vs_trust.png",dpi=160)
    plt.show()


In [ ]:
# 13. Manager vs IC trust
if {"ic_pm","trust_group"}.issubset(d.columns):
    tab = pd.crosstab(d["ic_pm"],d["trust_group"],normalize="index")*100
    tab.to_csv(OUTPUT/"manager_vs_ic_trust_pct.csv")
    display(tab.round(1))

    ax=tab.plot(kind="bar",figsize=(9,5))
    ax.set_ylabel("Percent")
    ax.set_title("AI trust: managers vs individual contributors")
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(FIGURES/"03_manager_vs_ic_trust.png",dpi=160)
    plt.show()
else:
    print("IC/PM field is not present in the 2025 dataset.")


In [ ]:
# 14. Geography
if {"country","trust_group"}.issubset(d.columns):
    top = d["country"].value_counts().head(12).index
    geo = d[d["country"].isin(top)]
    tab = pd.crosstab(geo["country"],geo["trust_group"],normalize="index")*100
    tab.to_csv(OUTPUT/"country_vs_trust_pct.csv")
    display(tab.round(1))

    if "Trust" in tab:
        tab["Trust"].sort_values().plot(kind="barh",figsize=(9,6))
        plt.xlabel("Trust (%)")
        plt.title("Trust in AI accuracy — largest respondent countries")
        plt.tight_layout()
        plt.savefig(FIGURES/"04_country_trust.png",dpi=160)
        plt.show()


In [ ]:
# 15. Export Power-BI-ready respondent dataset
export_cols = [c for c in [
    "id","country","age","work_exp","experience_band","dev_type","industry",
    "employment","remote_work","org_size","ic_pm","education",
    "ai_use","ai_sentiment","ai_accuracy","trust_group",
    "ai_complex","ai_threat","ai_benefit","ai_agents","ai_models","job_sat"
] if c in d.columns]

clean = d[export_cols].copy()
clean.to_csv(OUTPUT/"AI_Trust_Gap_Clean_Respondent_Data.csv",index=False)

print("DONE")
print("Rows:",len(clean))
print("Columns:",len(clean.columns))
print("Output folder:",OUTPUT)
print("\nFiles generated:")
for p in sorted(OUTPUT.rglob("*")):
    if p.is_file():
        print(" -",p.relative_to(OUTPUT))


## After Run All

Open the Colab **Files** panel and expand `outputs`.

You should now see:
- `AI_Trust_Gap_Clean_Respondent_Data.csv`
- `data_quality_report.csv`
- analysis CSVs
- `figures/` with PNG charts

Download those outputs and upload them back to ChatGPT for interpretation and dashboard development.
